# Ayiru × LangChain — agent savings demo

**What this shows.** A LangChain `AyiruTool` answering 10 developer-tool questions against a local Ayiru backend. We don't drive an LLM — we invoke the tool directly so the result is deterministic and runnable without API keys. Each cell prints USEFUL / WEAK / MISS, then the last cell pulls `/v1/stats/savings` and shows the token-savings tally.

**Honest framing.** The v0.2 bulk graph indexes 40 tools but **only 5 are curated to depth** (docker, git, github-cli, vercel-cli, openai-api). The other 35 are seeded with one thin index-page claim each — enough for `ask()` to return *something* with low confidence, but not enough for high-confidence answers. The demo deliberately splits results into:

- **USEFUL** — the top answer passes `is_useful` (`confidence ≥ 0.6 AND verification_level != "L0_unverified"`). This is what an agent should return to the user verbatim.
- **WEAK** — the server found a lexical match, but confidence or verification is too low. Agent should escalate.
- **MISS** — no match at all; agent must escalate to `web_search`.

**Preconditions.**
1. Ayiru backend running on `http://localhost:8000` (post-Stage-20 bulk graph: ~40 tools / ~82 claims).
2. `pip install -e clients/python[langchain]` from a checkout (PyPI publication is v0.2.5).

In [ ]:
import json

from ayiru_client import Answer, Ayiru
from ayiru_client.langchain import AyiruTool

BASE_URL = "http://localhost:8000"

client = Ayiru(base_url=BASE_URL)
tool = AyiruTool(client=client, return_response_json=True)  # full AskResponse for is_useful access

print(f"Tool ready — name={tool.name!r}")
print(f"Description (first 200 chars): {tool.description[:200]}…")

## 10-question batch

The first 4 questions target the 5 curated tools (deep coverage — we expect USEFUL answers). The next 3 target bulk-ingested tools (thin coverage — we expect WEAK answers, because the single index-page claim has low confidence). The last 3 are out-of-scope (expected MISS).

In [ ]:
QUESTIONS = [
    # --- expected USEFUL (curated, deep coverage) ---
    "how do I delete a github repo with gh",
    "what does git log do",
    "how do I list docker volumes",
    "how do I authenticate with the openai api",
    # --- expected WEAK (bulk-ingested, thin coverage — one index-page claim) ---
    "what does kubectl describe pod do",
    "how do I install a helm chart",
    "how do I install a package with apt",
    # --- expected MISS (out of scope for the v0.2 graph) ---
    "how do I configure my ergonomic keyboard",
    "what is the best programming language for embedded systems",
    "what is the airspeed velocity of an unladen swallow",
]

useful_count = weak_count = miss_count = 0

for i, q in enumerate(QUESTIONS, start=1):
    # return_response_json=True gives us the full AskResponse so we can
    # reconstruct an Answer and check `is_useful` properly.
    raw = tool.invoke({"question": q})
    payload = json.loads(raw)
    if payload["fallback_recommended"] or not payload["answers"]:
        miss_count += 1
        print(f"  [{i:>2}] MISS    →  {q}")
        continue
    top = Answer.model_validate(payload["answers"][0])
    if top.is_useful:
        useful_count += 1
        print(f"  [{i:>2}] USEFUL  →  {q}")
        print(f"             tool={top.tool_id}  conf={top.confidence:.2f}  level={top.verification_level}")
        print(f"             {top.statement[:140]}")
    else:
        weak_count += 1
        print(f"  [{i:>2}] WEAK    →  {q}")
        print(f"             tool={top.tool_id}  conf={top.confidence:.2f}  level={top.verification_level}  (below is_useful threshold)")

print(
    f"\nLocal tally: {useful_count} USEFUL / {weak_count} WEAK / {miss_count} MISS"
    f"  (target on v0.2 bulk graph: 4 / 3 / 3)"
)
print(
    "\nAgent guidance: return USEFUL verbatim; for WEAK/MISS, escalate to web_search."
)

## Server-side savings aggregate

Every `ask()` call above emitted a `QUERY_SERVED` audit event server-side. The `savings()` endpoint replays those events over a time window and reports total tokens saved (USEFUL answers contribute; WEAK and MISS don't, since the agent should escalate on those). This is the dashboard footer an agent team would put on their cost report.

In [ ]:
savings = client.savings("24h")

print(f"window:                   {savings.window}")
print(f"total queries served:     {savings.total_queries_served:,}")
print(f"  of those, fallbacks:    {savings.fallback_count:,}")
matched = savings.total_queries_served - savings.fallback_count
match_rate = matched / max(savings.total_queries_served, 1)
print(f"  match rate (any conf):  {match_rate:.0%}")
print(f"total tokens saved:       {savings.total_tokens_saved:,}")
print(f"estimated USD saved:      ${savings.estimated_usd_saved:.4f}")
print(f"  (priced at ${savings.usd_per_million_input_tokens}/M input tokens)")

print()
print(
    f"→ saved ~{savings.total_tokens_saved:,} tokens ≈ "
    f"${savings.estimated_usd_saved:.4f} across the last {savings.window}."
)
print(
    "Note: the savings aggregate counts every non-fallback hit, including "
    "WEAK ones. For an honest cost-savings number an agent team should "
    "only credit USEFUL hits — see the local tally above."
)

client.close()

## Next steps

To wire this into a real LangChain agent loop, replace the direct `tool.invoke(...)` calls with an agent that has `[AyiruTool(...), TavilySearch(...)]` in its tool list. The `description` field on `AyiruTool` is tuned to make the agent prefer Ayiru for stable technical questions; the agent should still fall through to web search on WEAK / MISS responses.

**Expanding coverage.** The bulk graph's WEAK results come from single-URL ingestion per tool. To make `kubectl`, `helm`, `apt` etc. produce USEFUL answers, add per-command URLs to [`tools/v0.2_seed_keep.json`](../../tools/v0.2_seed_keep.json) (e.g. `kubectl describe`, `kubectl logs`, …) and re-run `ayiru ingest --resume --tool-list …`. That's the planned v0.2.x depth work.

See the [Ayiru docs](https://github.com/ruth411/ayiru) for the architecture, verification ladder (L0–L5), and how to submit your own curated claims.